In [ ]:
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.config import BASE_DIR, FIGURES_DIR, NUMERICAL_COLS
from src.data_utils import load_raw_data, load_decoded_data
from src.preprocessing import split_train_test
from src.evaluation import evaluate_models_on_test, get_error_analysis_df

sns.set_theme(style="whitegrid", palette="muted")

# 1. Tải dữ liệu Test và các mô hình đã lưu từ Giai đoạn 4
df_raw = load_raw_data()
df_decoded = load_decoded_data()
X_train, X_test, y_train, y_test = split_train_test(df_raw)

models_dir = BASE_DIR / "models"
baseline_models = joblib.load(models_dir / "baseline_models.pkl")
tuned_models = joblib.load(models_dir / "tuned_models.pkl")

all_models = {**baseline_models, **tuned_models}

# 2. Chấm điểm trên tập Test và sắp xếp theo F1-Score giảm dần, Total_Cost tăng dần
test_results_df = evaluate_models_on_test(all_models, X_test, y_test)
test_results_sorted = test_results_df.sort_values(by=["Total_Cost", "F1_Score"], ascending=[True, False])

print("=== BẢNG TỔNG HỢP KẾT QUẢ TRÊN TẬP TEST (SẮP XẾP THEO CHI PHÍ LỖI TỐI ƯU) ===")
display(test_results_sorted.drop(columns=["Key"]).round(4))

# Lưu bảng kết quả ra CSV để đưa vào báo cáo LaTeX
reports_dir = BASE_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
test_results_sorted.drop(columns=["Key"]).round(4).to_csv(reports_dir / "test_evaluation_summary.csv", index=False)

In [ ]:
# 1. Vẽ Confusion Matrix cho 3 thuật toán đã tinh chỉnh trên Luồng B
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
stream_b_keys = ["Stream_B_K-NN", "Stream_B_Naive Bayes", "Stream_B_Decision Tree"]

for i, key in enumerate(stream_b_keys):
    model = all_models[key]
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=["Good (0)", "Bad (1)"],
        cmap="Blues", ax=axes[i], colorbar=False
    )
    axes[i].set_title(f"Confusion Matrix: {key.replace('Stream_B_', '')} (Luồng B)", fontweight="bold")
    axes[i].grid(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_confusion_matrices_stream_b.png", dpi=300)
plt.show()

# 2. Vẽ đường cong ROC so sánh Luồng A và Luồng B của 3 thuật toán
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for model_name in ["K-NN", "Naive Bayes", "Decision Tree"]:
    RocCurveDisplay.from_estimator(
        all_models[f"Stream_A_{model_name}"], X_test, y_test,
        name=model_name, ax=axes[0]
    )
    RocCurveDisplay.from_estimator(
        all_models[f"Stream_B_{model_name}"], X_test, y_test,
        name=model_name, ax=axes[1]
    )

axes[0].plot([0, 1], [0, 1], "k--", label="Ngẫu nhiên (AUC = 0.50)")
axes[0].set_title("Đường cong ROC trên Tập Test - Luồng A (Cơ bản)", fontweight="bold")
axes[0].legend(loc="lower right")

axes[1].plot([0, 1], [0, 1], "k--", label="Ngẫu nhiên (AUC = 0.50)")
axes[1].set_title("Đường cong ROC trên Tập Test - Luồng B (Cải tiến)", fontweight="bold")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "09_roc_curves_comparison.png", dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.barplot(
    data=test_results_df, x="Model", y="F1_Score", hue="Stream_Stage",
    ax=axes[0], palette="Set2"
)
axes[0].set_title("So sánh F1-Score trên Tập Test giữa các Phương án", fontweight="bold")
axes[0].set_ylabel("F1-Score (Lớp Bad = 1)")
axes[0].set_ylim(0, 0.75)

sns.barplot(
    data=test_results_df, x="Model", y="Total_Cost", hue="Stream_Stage",
    ax=axes[1], palette="Set1"
)
axes[1].set_title("Tổng Chi Phí Sai Số Tín Dụng trên Tập Test (Thấp hơn = Tốt hơn)", fontweight="bold")
axes[1].set_ylabel("Tổng điểm phạt (FP*1 + FN*5)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "10_test_metrics_and_cost.png", dpi=300)
plt.show()

In [ ]:
# Chọn mô hình có Tổng chi phí phạt (Total_Cost) thấp nhất trên tập Test để phân tích lỗi
best_key = test_results_sorted.iloc[0]["Key"]
best_model = all_models[best_key]
print(f"Mô hình được chọn để phân tích lỗi chi tiết: {best_key}")

err_df = get_error_analysis_df(best_model, X_test, y_test, df_decoded)

print("\n1. Thống kê số lượng các nhóm dự đoán trên tập Test:")
display(err_df["Error_Type"].value_counts().to_frame(name="Số lượng mẫu"))

print("\n2. So sánh giá trị trung bình các đặc trưng số giữa 4 nhóm dự đoán:")
display(err_df.groupby("Error_Type")[NUMERICAL_COLS + ["Prob_Bad"]].mean().round(2))

# 3. Trích xuất 5 hồ sơ Nợ Xấu bị bỏ sót nghiêm trọng nhất (FN: Thực tế Bad=1 nhưng dự đoán Good=0 với Prob_Bad rất thấp)
fn_cases = err_df[err_df["Error_Type"].str.startswith("FN")].sort_values(by="Prob_Bad", ascending=True)
cols_to_view = [
    "checking_status", "duration", "credit_history", "purpose",
    "credit_amount", "savings_status", "employment", "age", "Prob_Bad"
]
print("\n3. Top 5 hồ sơ Khách hàng Xấu bị dự đoán nhầm thành Tốt (Lỗi FN - Phạt 5 điểm/mẫu):")
display(fn_cases[cols_to_view].head(5))

# 4. Trích xuất 5 hồ sơ Khách hàng Tốt bị từ chối nhầm (FP: Thực tế Good=0 nhưng dự đoán Bad=1 với Prob_Bad rất cao)
fp_cases = err_df[err_df["Error_Type"].str.startswith("FP")].sort_values(by="Prob_Bad", ascending=False)
print("\n4. Top 5 hồ sơ Khách hàng Tốt bị dự đoán nhầm thành Xấu (Lỗi FP - Phạt 1 điểm/mẫu):")
display(fp_cases[cols_to_view].head(5))

# 5. Biểu đồ phân tích không gian lỗi theo Kỳ hạn vay (duration) và Số tiền vay (credit_amount)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=err_df, x="duration", y="credit_amount",
    hue="Error_Type", style="Error_Type", s=90, alpha=0.85,
    palette={
        "TN (Đúng khách Tốt)": "#2ecc71",
        "TP (Đúng khách Xấu)": "#3498db",
        "FP (Nhầm Tốt -> Xấu | Phạt 1)": "#f39c12",
        "FN (Sót Xấu -> Tốt | Phạt 5)": "#e74c3c"
    }
)
plt.title(f"Phân bố các trường hợp dự đoán Đúng/Sai ({best_key}) theo Kỳ hạn và Số tiền vay", fontweight="bold")
plt.xlabel("Thời hạn vay - Duration (Tháng)")
plt.ylabel("Số tiền vay - Credit Amount (DM)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "11_error_analysis_scatter.png", dpi=300)
plt.show()